# SHMS SIMC 蒙特卡洛预训练笔记本

本笔记本用于执行 Stage 1 预训练（SIMC Monte Carlo 数据）。
流程与仓库 `training/scripts/run_pretrain.py` 保持一致，并按数据交接约定组织输入变量与输出目标。

> 使用前请先修改 `SIMC_FILES` 为你的 ROOT 文件路径，并将 `OVERRIDE_P0` 设置为该批数据对应的中心动量（GeV/c）。

## 数据交接变量约定（SIMC -> 内部字段）

| SIMC branch | 内部字段 | 含义 |
|---|---|---|
| `hsxfp` | `x_fp` | 焦平面 x (cm) |
| `hsyfp` | `y_fp` | 焦平面 y (cm) |
| `hsxpfp` | `xp_fp` | 焦平面 x' (rad) |
| `hsypfp` | `yp_fp` | 焦平面 y' (rad) |
| `hsdeltai` | `delta` | 抛掷动量偏差 (%) |
| `hsxptari` | `xptar` | 靶面 θ (rad) |
| `hsyptari` | `yptar` | 靶面 φ (rad) |
| `hsztari` | `ytar` | 反应点 z (cm) |

补充特征：
- `x_tar`：按高斯噪声合成，标准差由 `x_tar_sigma_cm` 指定；
- `p0`：中心动量常量列（可固定或按配置抽样）。

In [ ]:
from __future__ import annotations

import glob
from pathlib import Path

import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt

from training.data.simc_dataset import SIMCDataset
from training.data.preprocessing import ScalerBundle
from training.models.residual_mlp import ResidualMLP
from training.models.physics_loss import PhysicsInformedLoss
from training.trainers.pretrain import PretrainTrainer

In [ ]:
# ===== 用户配置 =====
CONFIG_PATH = Path('../configs/pretrain_config.yaml')
SIMC_FILES = sorted(glob.glob('../../data/simc_run_*.root'))  # 示例：按项目内相对路径匹配
OUTPUT_DIR = Path('../../checkpoints/pretrain')
OVERRIDE_P0 = 4.4    # 示例值：请改成当前数据交接文件对应的中心动量（GeV/c）
MAX_EVENTS = None    # 例如 300000；None 表示不截断
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

assert CONFIG_PATH.exists(), f'Config not found: {CONFIG_PATH}'
assert len(SIMC_FILES) > 0, '请先配置 SIMC_FILES，至少匹配一个 ROOT 文件'
assert OVERRIDE_P0 is not None, '请设置 OVERRIDE_P0（例如 4.4）以启用 p0 输入列'

with open(CONFIG_PATH, 'r', encoding='utf-8') as fh:
    config = yaml.safe_load(fh)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config.setdefault('output', {})['checkpoint_dir'] = str(OUTPUT_DIR)
config['output']['scaler_save_path'] = str(OUTPUT_DIR / 'scaler_bundle.json')

print(f'Using device: {DEVICE}')
print(f'SIMC files: {len(SIMC_FILES)}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
tree_name = config.get('data', {}).get('simc_tree_name', 'h10')
x_tar_sigma = config.get('data', {}).get('x_tar_sigma_cm', 0.1)
seed = config.get('training', {}).get('random_seed', 42)

dataset = SIMCDataset(
    root_file_paths=SIMC_FILES,
    tree_name=tree_name,
    p0_value=OVERRIDE_P0,
    max_events=MAX_EVENTS,
    fit_scalers=True,
    x_tar_sigma_cm=x_tar_sigma,
    rng_seed=seed,
)

print(f'Dataset size: {len(dataset)}')
sample = dataset[0]
print('Input shape:', tuple(sample['inputs'].shape))
print('Target keys:', list(sample['targets'].keys()))

In [ ]:
has_p0 = int(sample['inputs'].shape[0]) == 6
input_features = ['x_fp', 'y_fp', 'xp_fp', 'yp_fp', 'x_tar'] + (['p0'] if has_p0 else [])
target_features = ['delta', 'xptar', 'yptar', 'ytar']
bundle = ScalerBundle(input_features=input_features, target_features=target_features)

if dataset.scaler_X is not None and dataset.scaler_Y is not None:
    bundle.set_fitted_scalers(dataset.scaler_X, dataset.scaler_Y)
    bundle.save(config['output']['scaler_save_path'])
    print('Saved scaler bundle to:', config['output']['scaler_save_path'])

In [ ]:
mcfg = config.get('model', {})
lcfg = config.get('loss', {})

model_input_dim = int(sample['inputs'].shape[0])
model = ResidualMLP(
    input_dim=model_input_dim,
    hidden_dim=mcfg.get('hidden_dim', 256),
    n_residual_blocks=mcfg.get('n_residual_blocks', 4),
    branch_dim=mcfg.get('branch_dim', 64),
)

loss_fn = PhysicsInformedLoss(
    lambda_physics=lcfg.get('lambda_physics', 0.01),
    use_huber=lcfg.get('use_huber', True),
    huber_delta=lcfg.get('huber_delta', 1.0),
    transport_matrix=lcfg.get('transport_matrix', None),
)

trainer = PretrainTrainer(model=model, loss_fn=loss_fn, config=config, device=DEVICE)
history = trainer.train(train_dataset=dataset, checkpoint_dir=str(OUTPUT_DIR))

print('Training completed.')
print('Best checkpoint:', OUTPUT_DIR / 'best_pretrain.pth')

In [ ]:
epochs = np.arange(1, len(history['train_loss']) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, history['train_loss'], label='train_loss')
plt.plot(epochs, history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('SIMC Pretraining Loss Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# 可选：快速检查 checkpoint 可加载性
ckpt_path = OUTPUT_DIR / 'best_pretrain.pth'
assert ckpt_path.exists(), f'Checkpoint not found: {ckpt_path}'
ckpt = torch.load(ckpt_path, map_location='cpu')
print('Saved epoch:', ckpt.get('epoch'))
print('Saved val_loss:', ckpt.get('val_loss'))
print('Checkpoint keys:', sorted(list(ckpt.keys())))